In [3]:
from enum import Enum
from dataclasses import dataclass
from pathlib import Path
import glob
import os
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("/data") / "irb" / "pharmacometrics" / "pro00117942"
HOVERNEXT_PREDICTIONS_DIR = DATA_DIR / "hover_next_predictions"
SLIDES_DIR = DATA_DIR / "digitized_slides"

class CellType(Enum):
    CONNECTIVE_TISSUE = "connective-tissue-cell"
    EOSINOPHIL = "eosinophil"
    EPITHELIAL_CELL = "epithelial-cell"
    LYMPHOCYTE = "lymphocyte"
    MITOSIS = "mitosis"
    NEUTROPHIL = "neutrophil"
    PLASMA_CELL = "plasma-cell"

@dataclass(eq=True, frozen=True)
class Cell:
    celltype : CellType
    x : float
    y : float

class PreAnnotatedSlide:
    def __init__(self, path_to_svs : Path, model_predictions_root : Path):
        if not path_to_svs.exists() or not model_predictions_root.exists(): raise ValueError("Please ensure both paths are valid and exist")
        self.path_to_svs = path_to_svs
        all_tsv_files_in_root = glob.glob("*.tsv", root_dir = model_predictions_root)
        if not all_tsv_files_in_root: raise ValueError(f"No TSV files found in {model_predictions_root}")
        
        all_cells_df = pd.read_csv(model_predictions_root / all_tsv_files_in_root[0], sep="\t")
        # TODO: refactor so concat isn't being called inside a loop
        # sue me...
        for tsv_file in all_tsv_files_in_root:
            this_cell_df = pd.read_csv(model_predictions_root / tsv_file, sep="\t")
            all_cells_df = pd.concat([all_cells_df, this_cell_df])

        self.cells = set()
        for _, row in all_cells_df.iterrows():
            cell = Cell(CellType(row["name"]), row["x"], row["y"])
            self.cells.add(cell)

# wrap in a list to eagerly evaluate
for slide_path in tqdm(list(SLIDES_DIR.iterdir())):
    corresponding_model_predictions_dir = HOVERNEXT_PREDICTIONS_DIR / slide_path.stem
    pre_annotated_slide = PreAnnotatedSlide(slide_path, corresponding_model_predictions_dir)

  0%|          | 0/546 [00:00<?, ?it/s]

/tmp/ipykernel_1474528/3953839688.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_cells_df = pd.concat([all_cells_df, this_cell_df])
/tmp/ipykernel_1474528/3953839688.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_cells_df = pd.concat([all_cells_df, this_cell_df])
/tmp/ipykernel_1474528/3953839688.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns whe

KeyboardInterrupt: 